# Naive RAG ハンズオン

## 1. PDFの読み込み

In [4]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from pprint import pprint

pdf_dir = "../../docs"

loader = PyPDFDirectoryLoader(pdf_dir)
data = loader.load()

pprint(data)

[Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': '新入社員研修マニュアル_FAQ付き', 'source': '../../docs/employee_training_manual_faq.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1'}, page_content='新 入 社 員 研 修 マ ニュ ア ル _FAQ 付 き  \n \n文 書 番 号 :  マ ニュ ア ル -2026-0001  \n施 行 日 :  2026 年 4 月 1 日  \n対 象 者 :  新 入 社 員、 中 途 入 社 者、 異 動 者  \n発 行 部 署 :  人 事 部  /  総 務 部  \n \n第 1 章 _ 会 社 概 要  株 式 会 社 ダ ミー テ ク ノ ロ ジー ズ へ よ う こ そ。  \n当 社 の ミッ ショ ン は、 グ ラ フ デー タ ベー ス 技 術 と 最 先 端 の 生 成 AI ア プ リ ケー ショ ン を 組 み 合 わ せ、\n新\nし\nい\nビ\nジ\nネ\nス\n価\n値\nを\n創\n造\nす\nる\nこ\nと\nで\nす。\n \n●  会 社 名 :  株 式 会 社 ダ ミー テ ク ノ ロ ジー ズ  ●  本 社 所 在 地 :  東 京 都 渋 谷 区 渋 谷 1-2-3  ●  主 要 事 業 :  AI シ ス テ ム 統 合、 ナ レッ ジ グ ラ フ コ ン サ ル ティ ン グ  ●  代 表 取 締 役 :  山 田  太 郎  \n第 2 章 _ 社 内 シ ス テ ム 利 用 ガ イ ド  日々 の 業 務 に 必 要 な 主 要 ツー ル お よ び シ ス テ ム の ア ク セ ス 手 順 で す。'),
 Document(metadata={'producer': 'Skia/PDF m153 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title'

## 2. ベクトルデータベース (埋め込み表現 / Chroma)

### 埋め込みモデル

In [6]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)

### Retriever (ベクトルデータベース: Chroma)

In [7]:
from langchain_chroma import Chroma

db = Chroma.from_documents(data, embeddings_model)

In [8]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [24]:
query = "社外に業務端末を持ち出す場合はどうすればいいですか？"

result = retriever.invoke(query)
print(result[0])

page_content='社 内 IT 機 器・ AI 利 用 ガ イ ド ラ イ ン  
文 書 番 号 :  REG-2026-0042  
制 定 日 :  2026 年 4 月 1 日  
最 終 改 定 日 :  2026 年 8 月 15 日  
主 管 部 署 :  情 報 シ ス テ ム 部  /  情 報 セ キュ リ ティ 委 員 会  
第 1 条 （目 的）  
本 規 定 は、 株 式 会 社 ダ ミー テ ク ノ ロ ジー ズ （以 下 「当 社」 と い う） に お け る 情 報 資 産 の 保 護 お よ び
業
務
効
率
化
を
両
立
さ
せ
る
た
め、
役
員、
従
業
員、
派
遣
社
員
（以
下
「従
業
員
等」
と
い
う）
が
利
用
す
る
社
内
IT
機
器
お
よ
び
生
成
AI
ツー
ル
の
安
全
な
運
用
基
準
を
定
め
る
こ
と
を
目
的
と
す
る。
 
第 2 条 （適 用 範 囲）  
本 規 定 は、 当 社 の 業 務 に 従 事 す る す べ て の 従 業 員 等、 な ら び に 当 社 が 保 有 ま た は 管 理 す る す
べ
て
の
情
報
機
器・
ネッ
ト
ワー
ク
環
境・
ソ
フ
ト
ウェ
ア
（ク
ラ
ウ
ド
サー
ビ
ス
含
む）
に
適
用
す
る。
 
第 3 条 （ IT 機 器 の 管 理 と 利 用 ルー ル）  
1.  パ ス ワー ド 管 理  ○  業 務 用 PC お よ び ア カ ウ ン ト の パ ス ワー ド は、 英 大 文 字・ 小 文 字・ 数 字・ 記 号 を 組
み
合
わ
せ
た
12
文
字
以
上
と
し、
他
者
と
の
共
有
を
固
く
禁
ず
る。
 2.  機 器 の 持 ち 出 し  ○  社 外 に 業 務 用 端 末 を 持 ち 出 す 場 合 は、 事 前 に 情 報 シ ス テ ム 部 へ 「端 末 持 ち 出 し
申
請
書」
を
提
出
し、
承
認
を
得
な
け
れ
ば
な
ら
な
い。
 3.  ソ フ ト ウェ ア の イ ン ス トー ル  ○  情 報 シ ス テ ム 部 が 許 可 し て い な い ソ フ ト ウェ ア （フ リー ソ フ ト 含 む） を

## 3. RAGを使用しない場合のLLMの回答

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [30]:
query = "社外に業務端末を持ち出す場合はどうすればいいですか？"
result = llm.invoke(query)

print(result.content[0]["text"])

社外に業務端末（PC、タブレット、スマートフォンなど）を持ち出す場合、企業の**「情報セキュリティポリシー（社内規程）」に従うことが大原則**です。

会社によって具体的な手順や許可レベルが異なりますが、一般的に踏むべき手順と注意すべきポイントを以下にまとめました。

---

### 1. 持ち出し前の手続き（事前準備）
勝手に持ち出すことは禁止されているケースがほとんどです。

*   **上長への申請・承認:**
    *   事前に「持ち出し申請書」の提出や、社内システムでの申請を行い、許可を得る。
    *   申請理由（出張、在宅勤務、客先訪問など）、持ち出し期間、行き先を明確にする。
*   **ルールの確認:**
    *   自社の「情報セキュリティ基本方針」や「モバイル端末管理規程」を再確認する。

### 2. 持ち出し時のセキュリティ対策（物理的・技術的対策）
紛失や盗難、情報漏洩を防ぐため、以下の対策が必須です。

*   **物理的な管理（絶対に肌身離さず持つ）:**
    *   移動中（電車、バス、タクシーなど）はバッグに入れ、網棚に置いたり足元に放置したりしない。
    *   飲食店やカフェなどでの**「置き引き」「ショルダーハッキング（画面の覗き見）」に十分注意**する。
    *   車の中に端末を置きっぱなしにしない（車上荒らし対策）。
    *   社外で作業する際は、プライバシーフィルター（覗き見防止フィルター）を使用する。
*   **技術的な対策（会社の指示に従う）:**
    *   **VPNの利用:** 社外から社内ネットワークや機密データにアクセスする際は、必ず指定のVPN（仮想プライベートネットワーク）を使用する。
    *   **公衆Wi-Fiの利用に注意:** セキュリティ対策が不十分な無料Wi-Fi（鍵マークがないものなど）は原則使用しない。使用する場合は会社のセキュアなWi-Fiやテザリング、モバイルルーターを使う。
    *   **画面ロックの設定:** 離席時は必ず画面ロック（Windowsなら `Win + L`）をかける。
    *   **暗号化・MDM:** 端末のハードディスクが暗号化されているか、MDM（モバイル端末管理）ツールが導入されているか確認する（通

## 4. Naive RAG を使用したLLM

### Retrieverの作成

In [31]:
from langchain_core.prompts.prompt import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["question", "context"],
    template="以下を参照して、質問に答えてください。\n\n{context}\n\n質問: {question}"
)

In [32]:
example = {"question": "This is question", "context": "This is context"}

result = prompt_template.invoke(example)
print(result)

text='以下を参照して、質問に答えてください。\n\nThis is context\n\n質問: This is question'


In [34]:
# ベクトルベータベースを作成
db = Chroma.from_documents(data, embeddings_model)

# 抽出器の作成
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 1})

In [35]:
# フォーマッタの作成
def text_formatter(retriever_output):
    """抽出器 (Retriever) による出力を整形する"""
    raw_text = retriever_output[0].page_content
    raw_text_to_newline = raw_text.replace("\n", "") # 改行文字を除外
    return raw_text_to_newline

In [36]:
query = "社外に業務端末を持ち出す場合はどうすればいいですか？"
retrieved = retriever.invoke(query)

print(text_formatter(retrieved))

社 内 IT 機 器・ AI 利 用 ガ イ ド ラ イ ン  文 書 番 号 :  REG-2026-0042  制 定 日 :  2026 年 4 月 1 日  最 終 改 定 日 :  2026 年 8 月 15 日  主 管 部 署 :  情 報 シ ス テ ム 部  /  情 報 セ キュ リ ティ 委 員 会  第 1 条 （目 的）  本 規 定 は、 株 式 会 社 ダ ミー テ ク ノ ロ ジー ズ （以 下 「当 社」 と い う） に お け る 情 報 資 産 の 保 護 お よ び業務効率化を両立させるため、役員、従業員、派遣社員（以下「従業員等」という）が利用する社内IT機器および生成AIツールの安全な運用基準を定めることを目的とする。 第 2 条 （適 用 範 囲）  本 規 定 は、 当 社 の 業 務 に 従 事 す る す べ て の 従 業 員 等、 な ら び に 当 社 が 保 有 ま た は 管 理 す る すべての情報機器・ネットワーク環境・ソフトウェア（クラウドサービス含む）に適用する。 第 3 条 （ IT 機 器 の 管 理 と 利 用 ルー ル）  1.  パ ス ワー ド 管 理  ○  業 務 用 PC お よ び ア カ ウ ン ト の パ ス ワー ド は、 英 大 文 字・ 小 文 字・ 数 字・ 記 号 を 組み合わせた12文字以上とし、他者との共有を固く禁ずる。 2.  機 器 の 持 ち 出 し  ○  社 外 に 業 務 用 端 末 を 持 ち 出 す 場 合 は、 事 前 に 情 報 シ ス テ ム 部 へ 「端 末 持 ち 出 し申請書」を提出し、承認を得なければならない。 3.  ソ フ ト ウェ ア の イ ン ス トー ル  ○  情 報 シ ス テ ム 部 が 許 可 し て い な い ソ フ ト ウェ ア （フ リー ソ フ ト 含 む） を 個 人 の 判 断 でインストールしてはならない。 第 4 条 （生 成 AI ツー ル の 利 用 基 準）  1.  利 用 可 能 な ツー ル  ○  業 務 に お い て 生 成 AI （ LLM 等） を 利 用 す る 場 合 は、 当 社 が 契 約・ 指 定 す る 公 式 環境（社内指定API経由環境等）のみを使用する

### RAGによる回答のチェイン作成

In [44]:
from langchain_core.runnables import RunnablePassthrough

In [38]:
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [39]:
chain = (
    {"question": RunnablePassthrough(), "context": retriever | text_formatter}
    | prompt_template
    | llm
)

In [43]:
query = "社外に業務端末を持ち出す場合はどうすればいいですか？"
result = chain.invoke(query)

print(result.content[0]["text"])

ご質問について、ガイドラインの「第3条（IT機器の管理と利用ルール） 2. 機器の持ち出し」に以下の通り定められています。

事前に情報システム部へ **「端末持ち出し申請書」を提出し、承認を得なければならない** とされています。
